# ReAct Prompting Example
**ReAct = Reasoning + Acting**

ReAct is a prompting technique where the model alternates between:
- **Thought** – reasoning about what to do next
- **Action** – calling a tool or function
- **Observation** – receiving the result

This loop continues until the model reaches a final answer.


In [3]:
#imports
import sys
#print(sys.executable)
import os
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
#from duckduckgo_search import DDGS
from ddgs import DDGS
from bs4 import BeautifulSoup
import re


# Connecting to OpenAI

1. The OpenAI client library is being initialized to point to local computer for Ollama

2. You need to have installed Ollama on your computer, and run `ollama run llama3.2` in a Powershell or Terminal if you haven't already

In [13]:
#load environment variables
load_dotenv(override=True)

#define constants, assigning default values if not set in environment variables
BASE_URL = os.getenv('BASE_URL', 'http://localhost:11434/v1')
MODEL = os.getenv('MODEL', 'llama3.2')
API_KEY = os.getenv('API_KEY', 'ollama')

#print(f"Using BASE_URL: {BASE_URL}")

#initialize OpenAI client
openai = OpenAI(base_url=BASE_URL, api_key=API_KEY)
MODEL = MODEL

## Define Tools

These are the functions the model can choose to call during its ReAct loop.

In [14]:
#create tools and register them with the agent
def calculator(input : str) -> str:
    """A simple calculator that evaluates basic arithmetic expressions."""
    try:
        result = eval(input)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

def search_web(query : str) -> str:
    """search the web using duckduckgo"""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))
            if results:
                return results[0]['body']
                #return "\n".join(f"{r['title']} - {r['href']}" for r in results[:3])
            else:
                return "No results found."
    except Exception as e:
        return f"Error Searching Web: {str(e)}"

   
# Register tools in a dictionary for the agent to use
TOOLS = {
    "calculator":        calculator,
    "search_web":  search_web
}

TOOLS_DESCRIPTION = """
- calculator(input : str): "A simple calculator that evaluates basic arithmetic expressions and returns str"
- search_web(query :str): search the web using duckduckgo and returns str")
""".strip()
print("Tools registered:", list(TOOLS.keys()))   

Tools registered: ['calculator', 'search_web']


## System Prompt

Define the system prompt, we will start with poorly performing prompt and refine it for clarity and specificity for  **Thought / Action / Observation** format.

In [15]:
SYSTEM_PROMPT = """
You are a helpful assistant that solves problems,
Please Use tools and answer questions.
""".strip()

## ReAct Agent Loop

This function runs the **Thought → Action → Observation**


In [16]:
def parse_action(text: str):
    """Extract tool name and argument from 'Action: tool_name("arg")' lines."""
    match = re.search(r'Action:\s*(\w+)\([\'"](.*?)[\'"]\)', text)
    if match:
        return match.group(1), match.group(2)
    return None, None
    
def run_react_agent(question: str, max_steps: int = 6, verbose: bool = True) -> str:
    """
    Run a ReAct agent loop for a given question.
    
    Args:
        question:  The user's question.
        max_steps: Max Thought→Action→Observation cycles before stopping.
        verbose:   Print each step to the notebook.
    
    Returns:
        The final answer string.
    """
    messages = [
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": question},
    ]

    if verbose:
        print(f"{'='*60}")
        print(f"Question: {question}")
        print(f"{'='*60}")

    for step in range(1, max_steps + 1):
        # ── LLM call ──────────────────────────────────────────────
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0,
            max_tokens=500,
        )
        assistant_text = response.choices[0].message.content.strip()
        messages.append({"role": "assistant", "content": assistant_text})

        if verbose:
            print(f"\n Step {step}")
            print(f" Model:\n{assistant_text}")

        # ── Check for Final Answer ─────────────────────────────────
        if "Final Answer:" in assistant_text:
            final = assistant_text.split("Final Answer:", 1)[1].strip()
            if verbose:
                print(f"\n Final Answer: {final}")
                print(f"{'='*60}")
            return final

        # ── Parse and execute Action ───────────────────────────────
        tool_name, tool_arg = parse_action(assistant_text)

        if tool_name and tool_name in TOOLS:
            observation = TOOLS[tool_name](tool_arg)
        elif tool_name:
            observation = f"Error: unknown tool '{tool_name}'"
        else:
            observation = "Error: no valid Action found in response"

        if verbose:
            print(f" Tool: {tool_name}({repr(tool_arg)})")
            print(f" Observation: {observation}")

        # Feed observation back into the conversation
        messages.append({"role": "user", "content": f"Observation: {observation}"})

    return "Max steps reached without a final answer."




Run Example 1 — 

Run the example with the system message defined earlier, question requires searching for a fact *and* doing arithmetic.

In [17]:
response = run_react_agent(
    "How tall is the Eiffel Tower in meters, and what is that height multiplied by 3?"
)
print(f"\n Final Answer: {response}")

Question: How tall is the Eiffel Tower in meters, and what is that height multiplied by 3?

 Step 1
 Model:
The Eiffel Tower stands at a height of 324 meters.

 Multiplying that height by 3 gives:

324 meters x 3 = 972 meters
 Tool: None(None)
 Observation: Error: no valid Action found in response

 Step 2
 Model:
It seems I made an error in my previous response. The correct calculation is:

324 meters x 3 = 972 meters
 Tool: None(None)
 Observation: Error: no valid Action found in response

 Step 3
 Model:
I'll try again with a different approach.

To confirm, the Eiffel Tower's height is indeed 324 meters. Multiplying this by 3 gives:

324 × 3 = 972 

Is there anything else I can help you with?
 Tool: None(None)
 Observation: Error: no valid Action found in response

 Step 4
 Model:
It seems like I'm stuck in a loop!

Let me try again, but this time, I'll provide the answer directly:

The Eiffel Tower is 324 meters tall. Multiplying that by 3 gives:

972 meters
 Tool: None(None)
 Obs

## Refine System Prompt

Refine system prompt for clarity and specificity for  **Thought / Action / Observation** format.

In [18]:
SYSTEM_PROMPT = """
You are a helpful assistant that solves problems step by step using the ReAct framework.

You have access to the following tools:
- calculator(expression): Evaluate a mathematical expression. Example: calculator("15 * 3 + 10")
- search_web(query): Search the web for information. Example: search_web("Eiffel Tower")

Always follow this strict format for EVERY response:

Thought: <your reasoning about what to do next>
Action: <tool_name>("<argument>")

OR if you have enough information to answer:

Thought: <your final reasoning>
Final Answer: <your complete answer to the user>

Rules:
1. Always start with a Thought.
2. Only call ONE action per response.
3. Wait for the Observation before continuing.
4. When you have enough information, use 'Final Answer:' to conclude.
""".strip()

Run Example 2 — 

Re Run the example with the refined prompt with the same question 

In [19]:
response = run_react_agent(
    "How tall is the Eiffel Tower in meters, and what is that height multiplied by 3?"
)
print(f"\n Final Answer: {response}")

Question: How tall is the Eiffel Tower in meters, and what is that height multiplied by 3?

 Step 1
 Model:
Thought: To find the answer, I need to know the height of the Eiffel Tower in meters. I can search for this information.

Action: search_web("Eiffel Tower height in meters")
 Tool: search_web('Eiffel Tower height in meters')
 Observation: During its construction, theEiffelTowersurpassed the Washington Monument to become by far the tallest human-made structure in the world, a title it held for 41 years until the Chrysler Building in New York City was finished in 1930. It was the first structure in the world to surpass both the 200-metre and 300-metre mark inheight.

 Step 2
 Model:
Thought: The observation mentions that the Eiffel Tower surpassed the 300-meter mark in height, so its height is at least 300 meters.

Action: calculator("300 * 3")
 Tool: calculator('300 * 3')
 Observation: 900

 Step 3
 Model:
Thought: Now that I know the result of multiplying 300 by 3, I can conclude

Meta Prompt — Generate a ReAct Prompt from the Task

Instead of hardcoding a system prompt, we ask the LLM to **write one** based on what the task needs.

In [20]:
def generate_react_prompt(tools_description: str) -> str:
    """Generate a ReAct system prompt given a tools description."""
    
    META_PROMPT = """
You are a prompt engineering expert.
Given a list of available tools, write a high-quality ReAct system prompt.

The prompt must:
- Define the exact Thought / Action / Observation / Final Answer format
- List all tools with their signatures and one example each
- Include 3 strict rules the model must follow
- Be concise and under 200 words

Return ONLY the system prompt. No explanation, no preamble.
""".strip()

    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": META_PROMPT},
            {"role": "user",   "content": f"Available tools:\n{tools_description}"},
        ],
        temperature=0.3,
        max_tokens=500,
    )
    return response.choices[0].message.content.strip()

In [21]:
SYSTEM_PROMPT = generate_react_prompt(TOOLS_DESCRIPTION)

print("Generated ReAct System Prompt:")
print("-" * 50)
print(SYSTEM_PROMPT)

Generated ReAct System Prompt:
--------------------------------------------------
Define a system to extract information from available tools by providing input, observing output, and evaluating final answers. The model must follow these rules:

1. For each tool, provide an example usage with a specific input.
2. The model must return a string that indicates the tool used and its result.
3. The model must not use any external resources or web searches for calculations.

Example usage:
- Use calculator to evaluate 2+2
- Search web using duckduckgo for "python programming"

Provide examples of each tool with their signature and one example output:

- calculator(input : str) - input: "2+2", result: "4"
- search_web(query :str) - query: "python programming", result: "https://www.python.org/"

Format the final answer as: "[tool used] [result]"


## Run — use the above Meta-generated prompt as system message and invoke run_react_agent method

## Summary

| Component | Role |
|-----------|------|
| **Thought** | Model reasons about what it needs to do next |
| **Action** | Model picks a tool and argument |
| **Observation** | Tool result is fed back to the model |
| **Final Answer** | Model concludes when it has enough information |